<a href="https://colab.research.google.com/github/peterbabulik/O-LLM_Holo/blob/main/01_Train_and_Export_OLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# STEP 1: O-LLM TRAINING & HOLOGRAPHIC FANO WEIGHT EXPORT (Google Colab)
# Architecture: 64D Octonionic Language Core (8 channels x 8D)
# Compression: 50% channel reduction via Fano Projective Closure (e0, e1, e2, e4)
# ==============================================================================

import json
import os
import time
import numpy as np
import scipy.linalg as la

print("[*] Launching Step 1: Training and Holographic Export of O-LLM...")

# ------------------------------------------------------------------------------
# 1. MATHEMATICAL ENGINE: RANK-3 FANO MULTIPLICATION TENSOR C_ijk (8x8x8)
# ------------------------------------------------------------------------------
def build_fano_tensor():
    """Builds the 8x8x8 multiplication tensor C_ijk for Octonions (O)."""
    C = np.zeros((8, 8, 8), dtype=np.float64)
    for i in range(8):
        C[0, i, i] = 1.0
        C[i, 0, i] = 1.0
    for i in range(1, 8):
        C[i, i, 0] = -1.0
    fano_triads = [
        (1, 2, 3), (1, 4, 5), (1, 7, 6),
        (2, 4, 6), (2, 5, 7), (3, 4, 7), (3, 6, 5)
    ]
    for i, j, k in fano_triads:
        C[i, j, k] =  1.0; C[j, k, i] =  1.0; C[k, i, j] =  1.0
        C[j, i, k] = -1.0; C[k, j, i] = -1.0; C[i, k, j] = -1.0
    return C

C8 = build_fano_tensor()

def oct_mul(a, b):
    """Multiplication of two 8D octonionic vectors along the Fano plane."""
    return np.einsum('i,j,ijk->k', a, b, C8)

def oct_assoc(a, b, c):
    """Ternary associator operator [a, b, c] = (a*b)*c - a*(b*c)."""
    ab_c = oct_mul(oct_mul(a, b), c)
    a_bc = oct_mul(a, oct_mul(b, c))
    return ab_c - a_bc

# ------------------------------------------------------------------------------
# 2. HOLOGRAPHIC FANO WEIGHT GENERATOR
# ------------------------------------------------------------------------------
def make_fano_octonion(c0, c1, c2, c4):
    """
    Generates a full 8D octonionic weight vector from only 4 generators (e0, e1, e2, e4).
    The remaining channels (e3, e5, e6, e7) are algebraically bound by the Fano lines:
      e3 = e1 * e2
      e5 = e1 * e4
      e6 = e2 * e4
      e7 = (e1 * e2) * e4
    """
    w = np.zeros(8, dtype=np.float64)
    w[0] = c0
    w[1] = c1
    w[2] = c2
    w[4] = c4
    w[3] = c1 * c2
    w[5] = c1 * c4
    w[6] = c2 * c4
    w[7] = c1 * c2 * c4
    norm = np.linalg.norm(w)
    return w / (norm + 1e-8)

# ------------------------------------------------------------------------------
# 3. DATA PREPARATION (TEXT CORPUS)
# ------------------------------------------------------------------------------
corpus_text = """to be or not to be that is the question whether tis nobler in the mind to suffer the slings and arrows of outrageous fortune or to take arms against a sea of troubles and by opposing end them to die to sleep no more and by a sleep to say we end the heart-ache and the thousand natural shocks that flesh is heir to tis a consummation devoutly to be wishd"""

# Vocabulary & token mappings
chars = sorted(list(set(corpus_text)))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}
V = len(chars)
T = len(corpus_text)

print(f"[*] Corpus loaded: {T} characters | Vocabulary: {V} unique tokens")

# Next-character prediction training pairs
X_seq = np.zeros((T - 1, V), dtype=np.float64)
Y_target = np.zeros((T - 1, V), dtype=np.float64)
for t in range(T - 1):
    X_seq[t, c2i[corpus_text[t]]] = 1.0
    Y_target[t, c2i[corpus_text[t + 1]]] = 1.0

# ------------------------------------------------------------------------------
# 4. MODEL ARCHITECTURE (64D O-Core)
# ------------------------------------------------------------------------------
nc = 8       # 8 octonionic phase channels
dim = nc * 8 # 64 total dimensions
rng = np.random.default_rng(42)

# Channel 2a control gate vector (symmetric vacuum carrier)
e_gate = np.array([1., 1, -1, 1, -1, 1, -1, 1], dtype=np.float64) / np.sqrt(8.0)

# Native Fano input projection weights: shape (V, nc, 4)
W_in_4d = rng.normal(0, 1.0, (V, nc, 4))
W_in_native = np.zeros((V, nc, 8), dtype=np.float64)
for v in range(V):
    for c in range(nc):
        W_in_native[v, c] = make_fano_octonion(*W_in_4d[v, c])

# Native Fano bypass weights: shape (nc, 4)
W_byp_4d = rng.normal(0, 1.0, (nc, 4))
W_byp_native = np.zeros((nc, 8), dtype=np.float64)
for c in range(nc):
    W_byp_native[c] = make_fano_octonion(*W_byp_4d[c])

# ------------------------------------------------------------------------------
# 5. MODEL TRAINING (Moiré Phase-Locking / Zero-Backpropagation)
# ------------------------------------------------------------------------------
t_train_start = time.time()

# Forward recurrence through the 64D Octonionic lattice
H_states = np.zeros((T - 1, dim), dtype=np.float64)
h = np.zeros((nc, 8), dtype=np.float64)
h[:, 0] = 1.0  # Vacuum identity ground state e0 = 1

alpha = 0.35   # Phase memory retention rate

for t in range(T - 1):
    xt = W_in_native[c2i[corpus_text[t]]]  # shape (nc, 8)
    h_next = np.zeros_like(h)
    for c in range(nc):
        # Non-linear associator [h, x, g] = (h*x)*g - h*(x*g)
        assoc = oct_assoc(h[c], xt[c], e_gate)
        assoc_norm = assoc / (np.linalg.norm(assoc) + 1e-6)

        # Pure octonionic bypass
        byp = oct_mul(xt[c], W_byp_native[c])

        # State transition on S^7 hypersphere
        step = (1.0 - alpha) * h[c] + alpha * assoc_norm + 0.3 * byp
        h_next[c] = step / (np.linalg.norm(step) + 1e-6)
    h = h_next
    H_states[t] = h.flatten()

# Output readout phase fitting via Ridge pseudoinverse
reg = 1e-3
W_out = la.pinv(H_states.T @ H_states + reg * np.eye(dim)) @ H_states.T @ Y_target

t_train_end = time.time()
print(f"[*] Model successfully trained in {t_train_end - t_train_start:.4f} seconds!")

# ------------------------------------------------------------------------------
# 6. EVALUATION OF THE TRAINED MODEL
# ------------------------------------------------------------------------------
logits = H_states @ W_out
preds = np.argmax(logits, axis=1)
targets = np.argmax(Y_target, axis=1)
acc = np.mean(preds == targets) * 100.0

exp_l = np.exp(logits - np.max(logits, axis=1, keepdims=True))
probs = exp_l / np.sum(exp_l, axis=1, keepdims=True)
loss = -np.mean(np.log(probs[np.arange(len(targets)), targets] + 1e-12))

print(f"\n" + "="*60)
print(f"[*] TRAINED MODEL PERFORMANCE (Pre-compression):")
print(f"    - Next-Character Accuracy: {acc:.2f} % (Random baseline: {100.0/V:.2f} %)")
print(f"    - Cross-Entropy Loss:      {loss:.4f}")
print("="*60)

# Sample text generation from prompt "to be "
prompt = "to be "
gen_text = prompt
h_gen = np.zeros((nc, 8), dtype=np.float64)
h_gen[:, 0] = 1.0

for ch in prompt:
    xt = W_in_native[c2i[ch]]
    h_next = np.zeros_like(h_gen)
    for c in range(nc):
        assoc = oct_assoc(h_gen[c], xt[c], e_gate)
        assoc_norm = assoc / (np.linalg.norm(assoc) + 1e-6)
        byp = oct_mul(xt[c], W_byp_native[c])
        step = (1.0 - alpha) * h_gen[c] + alpha * assoc_norm + 0.3 * byp
        h_next[c] = step / (np.linalg.norm(step) + 1e-6)
    h_gen = h_next

for _ in range(60):
    pred_logit = h_gen.flatten() @ W_out
    next_ch = i2c[np.argmax(pred_logit)]
    gen_text += next_ch

    xt = W_in_native[c2i[next_ch]]
    h_next = np.zeros_like(h_gen)
    for c in range(nc):
        assoc = oct_assoc(h_gen[c], xt[c], e_gate)
        assoc_norm = assoc / (np.linalg.norm(assoc) + 1e-6)
        byp = oct_mul(xt[c], W_byp_native[c])
        step = (1.0 - alpha) * h_gen[c] + alpha * assoc_norm + 0.3 * byp
        h_next[c] = step / (np.linalg.norm(step) + 1e-6)
    h_gen = h_next

print(f"[*] Generated text sample:\n    \"{gen_text}\"\n")

# ------------------------------------------------------------------------------
# 7. HOLOGRAPHIC COMPRESSION & DISK EXPORT
# ------------------------------------------------------------------------------
# A. Save full uncompressed model
np.savez_compressed("model_full.npz",
                    w_in=W_in_native,
                    w_byp=W_byp_native,
                    w_out=W_out)
size_full = os.path.getsize("model_full.npz")

# B. Holographic Compression: Save ONLY 4 generators (e0, e1, e2, e4)!
#    Channels e3, e5, e6, e7 are completely discarded (50% phase tensor reduction).
np.savez_compressed("model_compressed_holographic.npz",
                    w_in_4d=W_in_4d,
                    w_byp_4d=W_byp_4d,
                    w_out=W_out)
size_comp = os.path.getsize("model_compressed_holographic.npz")

# C. Save metadata for Step 2 (tokenizer mappings & network hyperparameters)
metadata = {
    "chars": chars,
    "c2i": c2i,
    "nc": nc,
    "dim": dim,
    "alpha": alpha,
    "prompt_sample": prompt,
    "accuracy": float(acc),
    "loss": float(loss)
}
with open("metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("="*60)
print(f"[*] HOLOGRAPHIC EXPORT COMPLETED:")
print(f"    1. Full Model:            'model_full.npz'                  ({size_full:,} B)")
print(f"    2. Compressed Model:      'model_compressed_holographic.npz' ({size_comp:,} B)")
print(f"    3. Tokenizer Metadata:    'metadata.json'")
print(f"    -> Storage footprint reduction: {(1.0 - size_comp / size_full) * 100:.2f} % (50% in phase weight tensors)")
print("="*60)
print("[+] Files are saved in the current Colab working directory.")

[*] Launching Step 1: Training and Holographic Export of O-LLM...
[*] Corpus loaded: 353 characters | Vocabulary: 25 unique tokens
[*] Model successfully trained in 0.2017 seconds!

[*] TRAINED MODEL PERFORMANCE (Pre-compression):
    - Next-Character Accuracy: 62.78 % (Random baseline: 4.00 %)
    - Cross-Entropy Loss:      2.9236
[*] Generated text sample:
    "to be or on to t at tiee sa ht he t to to thasi nd ae t to s hat l"

[*] HOLOGRAPHIC EXPORT COMPLETED:
    1. Full Model:            'model_full.npz'                  (25,670 B)
    2. Compressed Model:      'model_compressed_holographic.npz' (19,326 B)
    3. Tokenizer Metadata:    'metadata.json'
    -> Storage footprint reduction: 24.71 % (50% in phase weight tensors)
[+] Files are saved in the current Colab working directory, ready for Step 2.
